#### Sensor faults/malfunctions simulation
<small>
CONTENT: Definition of methods for sensor malfunctions and faults and dataset contamination <br>
</small>

#### Setup

In [ ]:
# IMPORT
from math import sqrt
import os
import re
from statistics import *
import numpy as np
import pandas as pd
import pywt
import h5py
import scipy.stats as stats
from scipy.signal import butter, filtfilt, welch
import random
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as ticker
from datetime import datetime, timedelta

In [ ]:
# FOLDER
mainFolder = os.getcwd()
inputFolder = os.path.join(mainFolder, "input")
outputFolder = os.path.join(mainFolder, "output")

if not os.path.exists(inputFolder):
	os.makedirs(inputFolder)
if not os.path.exists(outputFolder):
	os.makedirs(outputFolder)

#### Input

In [ ]:
# TIMESERIES FOR PLOTS
startDate = datetime(2020,1,1,0,0,0) 		# Start date of recording: 01/01/2020 00:00
endDate = datetime(2022,12,31,23,0,0) 		# End date of recording: 31/12/2023 23:00
intVal = 1 									# Interval of acquisition
offsetAlias = 'H'							# Unit of measument of acquisition interval [hours]
tHourly = pd.date_range(start = startDate, end = endDate, freq = ('{}{}'.format(intVal, offsetAlias)))
# erase feb 29 in 2020 and 2024
timeSimulation = tHourly[~((tHourly.month == 2) & (tHourly.day == 29))]
daysMonitored = len(pd.Series(timeSimulation.date).unique())
endDatePlot = datetime(2023,1,1,0,1,0)

In [ ]:
# SIMULATION INPUT
days = 365
hours = 24
years = 3
hours1year = days * hours
daysTot = days * years
timeSteps = days * hours * years
timeSeries = np.linspace(0, timeSteps-1, timeSteps)
nSim = 5 # maximum number of simulations for data generation
timeSeriesMax = np.linspace(0, (timeSteps * nSim)-1, (timeSteps * nSim))
# units kg,m,s
time = np.arange(0, daysTot, 1/hours)

In [ ]:
# LOAD ALL ACCELERATION DATA FILENAMES
accSFMfolder = os.path.join(mainFolder, "output", "acceleration", "D1")
accList = os.listdir(accSFMfolder)
accFiles = [os.path.join(accSFMfolder, entry) for entry in accList]
accD5 = os.path.join(mainFolder, "output", "acceleration", "D5")
accListD5 = os.listdir(accD5)
accFilesD5 = [os.path.join(accD5, entry) for entry in accListD5]

#### Import reference acceleration signal

In [ ]:
# TIME VECTOR
fs = 100 # Hz
minutes = 3
acquisLength = minutes * 60  # acquisition length in seconds (5mins)
nSamples = (fs * acquisLength)  # total number of samples
dt = 1/fs
t = np.linspace(0, acquisLength, nSamples)  # time vector

In [ ]:
# IMPORT FIRST ACC
firstAcc = os.path.join(mainFolder, "output", "acceleration", "D1", 'acc00000-1.h5')
with h5py.File(firstAcc, 'r') as f:
	acc = f['acc'][:]
	acc = acc[0]

In [ ]:
# PLOT EXAMPLE REFERENCE + SPECTRUM
def extractFreq(acc,fs,nperseg,noverlap):
	freqs, psd = welch(acc, fs, window = 'hann', nperseg = nperseg, noverlap = noverlap)
	peakInd = np.argmax(psd)
	extractedFreq = freqs[peakInd]
	return freqs, psd, extractedFreq
nperseg = 2**10
noverlap = nperseg *0.5
fwRef, psdRef, freqRef = extractFreq(acc,fs,nperseg,noverlap)

# Plot
plt.rcParams['text.usetex'] = True
plt.rcParams['figure.dpi'] = 300
plt.rcParams['font.size'] = 16

# single figure with time series and spectrum
fig = plt.figure(figsize=(16, 2.5))
gs = mpl.gridspec.GridSpec(1, 2, width_ratios=[2, 1]) 
fig.suptitle('0 - Reference (no faults or malfunctions)', fontsize = 20, y = 1.05)

# Total Load Time Series
ax1 = fig.add_subplot(gs[0, 0])

# time history
ax1.plot(t, acc, color = 'grey', linewidth = 0.3, alpha = 0.5)
ax1.plot(t, acc, color = 'black', linewidth = 0.3)
ax1.set_xlabel('$t$ [$s$]', fontsize = 18)
ax1.set_ylabel('$a_{ref}$ $[m/s^{2}]$')
ax1.set_xlim(t[0], t[-1])
ax1.tick_params(which ='major', direction = 'in', length = 10, labelsize = 14, pad = 5)  # Adjust major tick size
ax1.tick_params(which ='minor', direction = 'in', length = 5, pad = 5)  # Minor ticks are smaller
plt.grid(True)

# spectrum
ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(fwRef[:-1], psdRef[:-1], linewidth=1, color="black")
ax2.axvline(x=freqRef, color='black', linestyle='--', linewidth=1.5)
ax2.text(freqRef + 0.3, 0.55e-6, f'f = {freqRef:.2f} Hz', color='black', fontsize=14)
ax2.set_xlim(0, 50)
ax2.set_ylim(0, 0.8e-6)
ax2.set_xlabel(r'$f$ [$Hz$]')
ax2.set_ylabel(r'$PSD$ [$m^2/s^3$]')
ax2.tick_params(which='major', direction='in', length=10, labelsize=18, pad=10)  # Adjust major tick size
ax2.tick_params(which='minor', direction='in', length=5, pad=10)  # Minor ticks are smaller
plt.grid(True)
plt.show()

#### Sensor fault simulations methods and examples

In [ ]:
# DEFINE RANGES OF VALUES FOR SENSOR FAULT/MALFUNCTIONS SIMULATION
# same for all
begLimit = 20
endLimit = 20
startPoint = np.random.randint(begLimit, acquisLength-endLimit, size = 1)
numFilesMultiIndexFault = (24*2,24*10)

# SF/M specific
# drift
scaleDriftRange = (1e-3, 2e-3)
# spike
spikeMaxRange = (5e-2, 5e-3)
spikeNumRange = (5,20)
# bias
biasMagRange = (1.2,1.7)
# gain
factorGainRange = (1.1,3.1)
# noise
noiseStdRange = (1e-3, 5e-3)
# missing
keepPercMissingRange = (0.2, 0.8)
# cable detachment
duration = 2500
frequencyRange = (0.5,1.5)
amplitudeRange = (0.001,0.005)
damping_factor = 0.05

In [ ]:
# SENSOR FAULT/MALFUNCTION SIMULATION METHODS
# DRIFT
def simulateDrift(accelerogram, startPoint, fs, scale = random.uniform(*scaleDriftRange)):
	# start point is in seconds transform it into index
	driftedAcc = np.zeros_like(accelerogram)
	startIndex = startPoint * fs
	n = len(accelerogram)
	t = np.arange(n-startIndex)
	# exponential drift
	mean = np.mean(accelerogram)
	drift = - scale * np.exp(-t / (n / 10))
	# Apply exponential drift starting from the specified start point
	driftedAcc[:startIndex] = accelerogram[:startIndex]
	driftedAcc[startIndex:] = accelerogram[startIndex:] - drift - scale
	driftShift = driftedAcc[-1]
	return driftedAcc, driftShift  # Start drift effect from start_point and keep it continuous

def simulateFullDrift(accelerogram, driftShift):
	driftShiftedAcc = accelerogram.copy()
	driftShiftedAcc += driftShift
	return driftShiftedAcc

# SPIKE
def simulateSpike(accelerogram, startPoint, spikeMax = random.uniform(*spikeMaxRange)):
	spikedAcc = accelerogram.copy()
	n = len(acc)
	startIndex = startPoint * fs
	endIndex = startIndex + 1
	spikedAcc[startIndex] += spikeMax
	spikedAcc[endIndex] -= spikeMax
	return spikedAcc

# GAIN
def simulateGain(accelerogram, startPoint, fs, factorGain = random.uniform(*factorGainRange)):
	gainedAcc = accelerogram.copy() 
	startIndex = startPoint * fs
	gainedAcc[:startIndex] = accelerogram[:startIndex]
	gainedAcc[startIndex:] = accelerogram[startIndex:] * factorGain
	return gainedAcc, factorGain

def simulateFullGain(accelerogram, factorGain):
	fullGainAcc = accelerogram.copy()
	fullGainAcc = fullGainAcc * factorGain
	return fullGainAcc

#BIAS
def simulateBias(accelerogram, startPoint, fs, biasMag = random.uniform(*biasMagRange)):
	biasedAcc = accelerogram.copy()
	startIndex = int(startPoint * fs)
	biasedAcc[startIndex:] += np.std(acc)*biasMag
	return biasedAcc, biasMag

def simulateFullBias(accelerogram, biasMag):
	fullBiasedAcc = accelerogram.copy()
	fullBiasedAcc += np.std(acc)*biasMag
	return fullBiasedAcc

# NOISE
def simulateNoise50(accelerogram, startPoint, fs, noiseStd = random.uniform(*noiseStdRange), noiseFreq = 50):
	# Create a time array for the accelerogram
	t = np.arange(0, len(accelerogram)) / fs
	# Generate sinusoidal noise at the specified frequency (e.g., 50 Hz)
	phase = np.random.uniform(0, 2*np.pi)  # Random phase for the sinusoidal noise
	noise = noiseStd * np.sin(2 * np.pi * noiseFreq * t + phase)
	# Copy accelerogram to apply noise
	noisedAcc = accelerogram.copy()
	# Calculate the index in the array where to start applying noise
	startIndex = int(startPoint * fs)
	# Add the noise starting at the specified index
	noisedAcc[startIndex:] = accelerogram[startIndex:] + noise[startIndex:]
	return noisedAcc, noise

def simulateFullNoise50(accelerogram, noise):
	noisedFullAcc = accelerogram.copy()
	# Add the noise starting at the specified index
	noisedFullAcc = accelerogram + noise
	return noisedFullAcc

# MISSING DATA
def simulateMissing(accelerogram, keepPercMissing = random.uniform(*keepPercMissingRange)):
	start = int(keepPercMissing * len(accelerogram))
	missingAcc = accelerogram.copy()
	missingAcc[start:] = 0.000001
	return missingAcc

def simulateFullMissing(accelerogram):
	missingFullAcc = accelerogram.copy()
	missingFullAcc = 0.000001
	return missingFullAcc

# CABLE DETACHMENT
def simulateCableDetachReattach(accelerogram, startPoint, fs, duration = 850, 
						frequency = random.uniform(*frequencyRange), amplitude = random.uniform(*amplitudeRange), 
						damping_factor = 0.05):
	n = len(accelerogram)
	timeS = np.arange(0,duration,1/fs)
	startIndex = startPoint * fs
	omega = 2 * np.pi * frequency
	detach = amplitude * np.sin(omega * timeS) * np.exp(-damping_factor * omega * timeS)
	accelerogramDetach = accelerogram.copy()
	#print(len(accelerogramDetach[startIndex:startIndex+duration]))
	#print(len(detach[:duration]))
	accelerogramDetach[startIndex:startIndex+duration] += detach[:duration]
	return accelerogramDetach

def simulateCableDetach(accelerogram, startPoint, fs, duration = 850, 
						frequency = random.uniform(*frequencyRange), amplitude = random.uniform(*amplitudeRange), 
						damping_factor = 0.05):
	n = len(accelerogram)
	timeS = np.arange(0,duration,1/fs)
	startIndex = startPoint * fs
	omega = 2 * np.pi * frequency
	detach = amplitude * np.sin(omega * timeS) * np.exp(-damping_factor * omega * timeS)
	accelerogramDetach = accelerogram.copy()
	#print(len(accelerogramDetach[startIndex:startIndex+duration]))
	#print(len(detach[:duration]))
	accelerogramDetach[startIndex:(startIndex+duration)] += detach[:duration]
	accelerogramDetach[(startIndex+duration):] = 0.000001
	return accelerogramDetach

In [ ]:
# PLOT EXAMPLE DRIFT
# example
startPoint = np.random.randint(20, acquisLength-20, size = 1)
startPoint = startPoint[0]
aD, driftShift = simulateDrift(acc, startPoint, fs = fs, scale = 0.005) # start Point is provided in seconds

accPlot = aD
def extractFreq(acc,fs,nperseg,noverlap):
	freqs, psd = welch(acc, fs, window = 'hann', nperseg = nperseg, noverlap = noverlap)
	peakInd = np.argmax(psd)
	extractedFreq = freqs[peakInd]
	return freqs, psd, extractedFreq
nperseg = 2**10
noverlap = nperseg *0.5
fwPlot, psdPlot, freq = extractFreq(accPlot,fs,nperseg,noverlap)

# Plot
plt.rcParams['text.usetex'] = True
plt.rcParams['figure.dpi'] = 300
plt.rcParams['font.size'] = 16

# single figure with time series and spectrum
fig = plt.figure(figsize=(16, 2.5))
gs = mpl.gridspec.GridSpec(1, 2, width_ratios=[2, 1]) 
fig.suptitle('1 - Drift (D)', fontsize = 20, y = 1.05)

# Total Load Time Series
ax1 = fig.add_subplot(gs[0, 0])

# time history
ax1.plot(t, acc, color = 'grey', linewidth = 0.3, alpha = 0.5)
ax1.plot(t, accPlot, color = 'black', linewidth = 0.3)
ax1.set_xlabel('$t$ [$s$]', fontsize = 18)
ax1.set_ylabel('$a_{D}$ $[m/s^{2}]$')
ax1.set_xlim(t[0], t[-1])
ax1.tick_params(which ='major', direction = 'in', length = 10, labelsize = 14, pad = 5)  # Adjust major tick size
ax1.tick_params(which ='minor', direction = 'in', length = 5, pad = 5)  # Minor ticks are smaller
plt.grid(True)

# spectrum
ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(fwRef[:-1], psdRef[:-1], linewidth=1, color="grey")
ax2.plot(fwPlot[:-1], psdPlot[:-1], linewidth=1, color="black")
ax2.axvline(x=freq, color='black', linestyle='--', linewidth=1.5)
ax2.text(freq + 0.3, 0.55e-6, f'f = {freq:.2f} Hz', color='black', fontsize=14)
ax2.set_xlim(0, 50)
ax2.set_ylim(0, 0.8e-6)
ax2.set_xlabel(r'$f$ [$Hz$]')
ax2.set_ylabel(r'$PSD$ [$m^2/s^3$]')
ax2.tick_params(which='major', direction='in', length=10, labelsize=14, pad=10)  # Adjust major tick size
ax2.tick_params(which='minor', direction='in', length=5, pad=10)  # Minor ticks are smaller
plt.grid(True)
plt.show()

In [ ]:
# PLOT EXAMPLE SHIFT AFTER DRIFT
# example
aDF = simulateFullDrift(acc, driftShift)

accPlot = aDF
def extractFreq(acc,fs,nperseg,noverlap):
	freqs, psd = welch(acc, fs, window = 'hann', nperseg = nperseg, noverlap = noverlap)
	peakInd = np.argmax(psd)
	extractedFreq = freqs[peakInd]
	return freqs, psd, extractedFreq
nperseg = 2**10
noverlap = nperseg *0.5
fwPlot, psdPlot, freq = extractFreq(accPlot,fs,nperseg,noverlap)

# Plot
plt.rcParams['text.usetex'] = True
plt.rcParams['figure.dpi'] = 300
plt.rcParams['font.size'] = 16

# single figure with time series and spectrum
fig = plt.figure(figsize=(16, 2.5))
gs = mpl.gridspec.GridSpec(1, 2, width_ratios=[2, 1]) 
fig.suptitle('2.1 - Bias/Shift after Drift (B)', fontsize = 20, y = 1.05)

# Total Load Time Series
ax1 = fig.add_subplot(gs[0, 0])

# time history
ax1.plot(t, acc, color = 'grey', linewidth = 0.3, alpha = 0.5)
ax1.plot(t, accPlot, color = 'black', linewidth = 0.3)
ax1.set_xlabel('$t$ [$s$]', fontsize = 18)
ax1.set_ylabel('$a_{DS}$ $[m/s^{2}]$')
ax1.set_xlim(t[0], t[-1])
ax1.tick_params(which ='major', direction = 'in', length = 10, labelsize = 14, pad = 5)  # Adjust major tick size
ax1.tick_params(which ='minor', direction = 'in', length = 5, pad = 5)  # Minor ticks are smaller
plt.grid(True)

# spectrum
ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(fwRef[:-1], psdRef[:-1], linewidth=1, color="grey")
ax2.plot(fwPlot[:-1], psdPlot[:-1], linewidth=1, color="black")
ax2.axvline(x=freq, color='black', linestyle='--', linewidth=1.5)
ax2.text(freq + 0.3, 0.55e-6, f'f = {freq:.2f} Hz', color='black', fontsize=14)
ax2.set_xlim(0, 50)
ax2.set_ylim(0, 0.8e-6)
ax2.set_xlabel(r'$f$ [$Hz$]')
ax2.set_ylabel(r'$PSD$ [$m^2/s^3$]')
ax2.tick_params(which='major', direction='in', length=10, labelsize=14, pad=10)  # Adjust major tick size
ax2.tick_params(which='minor', direction='in', length=5, pad=10)  # Minor ticks are smaller
plt.grid(True)
plt.show()

In [ ]:
# PLOT EXAMPLE BIAS
# example
startPoint = np.random.randint(20, acquisLength-20, size = 1)
startPoint = startPoint[0]
aBi, _ = simulateBias(acc, startPoint, fs, biasMag = 1.8)

accPlot = aBi
def extractFreq(acc,fs,nperseg,noverlap):
	freqs, psd = welch(acc, fs, window = 'hann', nperseg = nperseg, noverlap = noverlap)
	peakInd = np.argmax(psd)
	extractedFreq = freqs[peakInd]
	return freqs, psd, extractedFreq
nperseg = 2**10
noverlap = nperseg *0.5
fwPlot, psdPlot, freq = extractFreq(accPlot,fs,nperseg,noverlap)

# Plot
plt.rcParams['text.usetex'] = True
plt.rcParams['figure.dpi'] = 300
plt.rcParams['font.size'] = 16

# single figure with time series and spectrum
fig = plt.figure(figsize=(16, 2.5))
gs = mpl.gridspec.GridSpec(1, 2, width_ratios=[2, 1]) 
fig.suptitle('2.2 - Bias/Shift (B)', fontsize = 20, y = 1.05)

# Total Load Time Series
ax1 = fig.add_subplot(gs[0, 0])

# time history
ax1.plot(t, acc, color = 'grey', linewidth = 0.3, alpha = 0.5)
ax1.plot(t, accPlot, color = 'black', linewidth = 0.3)
ax1.set_xlabel('$t$ [$s$]', fontsize = 18)
ax1.set_ylabel('$a_{B}$ $[m/s^{2}]$')
ax1.set_xlim(t[0], t[-1])
ax1.tick_params(which ='major', direction = 'in', length = 10, labelsize = 14, pad = 5)  # Adjust major tick size
ax1.tick_params(which ='minor', direction = 'in', length = 5, pad = 5)  # Minor ticks are smaller
plt.grid(True)

# spectrum
ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(fwRef[:-1], psdRef[:-1], linewidth=1, color="grey")
ax2.plot(fwPlot[:-1], psdPlot[:-1], linewidth=1, color="black")
ax2.axvline(x=freq, color='black', linestyle='--', linewidth=1.5)
ax2.text(freq + 0.3, 0.55e-6, f'f = {freq:.2f} Hz', color='black', fontsize=14)
ax2.set_xlim(0, 50)
ax2.set_ylim(0, 0.8e-6)
ax2.set_xlabel(r'$f$ [$Hz$]')
ax2.set_ylabel(r'$PSD$ [$m^2/s^3$]')
ax2.tick_params(which='major', direction='in', length=10, labelsize=14, pad=10)  # Adjust major tick size
ax2.tick_params(which='minor', direction='in', length=5, pad=10)  # Minor ticks are smaller
plt.grid(True)
plt.show()

In [ ]:
# PLOT EXAMPLE GAIN
# example
startPoint = np.random.randint(20, acquisLength-20, size = 1)
startPoint = startPoint[0]
aG, _ = simulateGain(acc, startPoint, fs = fs, factorGain = 5.3) # start Point is provided in seconds

accPlot = aG
def extractFreq(acc,fs,nperseg,noverlap):
	freqs, psd = welch(acc, fs, window = 'hann', nperseg = nperseg, noverlap = noverlap)
	peakInd = np.argmax(psd)
	extractedFreq = freqs[peakInd]
	return freqs, psd, extractedFreq
nperseg = 2**10
noverlap = nperseg *0.5
fwPlot, psdPlot, freqPlot = extractFreq(accPlot,fs,nperseg,noverlap)


psdPlotDb = 10 * np.log10(psdPlot)
psdRefDb = 10 * np.log10(psdRef)

# Plot
plt.rcParams['text.usetex'] = True
plt.rcParams['figure.dpi'] = 300
plt.rcParams['font.size'] = 16

# single figure with time series and spectrum
fig = plt.figure(figsize=(16, 2.5))
gs = mpl.gridspec.GridSpec(1, 2, width_ratios=[2, 1]) 
fig.suptitle('3 - Gain (G)', fontsize = 20, y = 1.05)

# Total Load Time Series
ax1 = fig.add_subplot(gs[0, 0])

# time history
ax1.plot(t, acc, color = 'grey', linewidth = 0.3, alpha = 0.5)
ax1.plot(t, accPlot, color = 'black', linewidth = 0.3)
ax1.set_xlabel('$t$ [$s$]', fontsize = 18)
ax1.set_ylabel('$a_{G}$ $[m/s^{2}]$')
ax1.set_xlim(t[0], t[-1])
ax1.tick_params(which ='major', direction = 'in', length = 10, labelsize = 14, pad = 5)  # Adjust major tick size
ax1.tick_params(which ='minor', direction = 'in', length = 5, pad = 5)  # Minor ticks are smaller
plt.grid(True)

# spectrum
ax2 = fig.add_subplot(gs[0, 1])
#ax2.plot(fwRef[10:], psdRefDb[10:], linewidth = 1, color = "grey")
#ax2.plot(fwPlot[10:], psdPlotDb[10:], linewidth = 1, color = "black")
#ax2.set_ylabel(r'$PSD$ [$dB$]')

ax2.plot(fwRef[:-1], psdRef[:-1], linewidth=1, color="grey")
ax2.plot(fwPlot[:-1], psdPlot[:-1], linewidth=1, color="black")
ax2.axvline(x=freqRef, color='grey', linestyle='--', linewidth=1.5)
#ax2.axvline(x=freqPlot, color='black', linestyle='--', linewidth=1.5)
ax2.text(freqRef + 0.3, 0.55e-6, f'f = {freqRef:.2f} Hz', color='grey', fontsize=14)
#ax2.text(freqPlot - 15, 3e-6, f'f = {freqPlot:.2f} Hz', color='black', fontsize=14)
ax2.set_xlim(0, 50)
#ax2.set_ylim(0, 0.8e-6)
ax2.set_xlabel(r'$f$ [$Hz$]')
ax2.set_ylabel(r'$PSD$ [$m^2/s^3$]')
ax2.tick_params(which='major', direction='in', length=10, labelsize=14, pad=10)  # Adjust major tick size
ax2.tick_params(which='minor', direction='in', length=5, pad=10)  # Minor ticks are smaller
plt.grid(True)
plt.show()

In [ ]:
# PLOT EXAMPLE NOISE
# example
startPoint = np.random.randint(20, acquisLength-20, size = 1)
startPoint = startPoint[0]
aN, _ = simulateNoise50(acc, startPoint, fs = fs, noiseStd = 0.01) # start Point is provided in seconds

accPlot = aN
def extractFreq(acc,fs,nperseg,noverlap):
	freqs, psd = welch(acc, fs, window = 'hann', nperseg = nperseg, noverlap = noverlap)
	peakInd = np.argmax(psd)
	extractedFreq = freqs[peakInd]
	return freqs, psd, extractedFreq
nperseg = 2**10
noverlap = nperseg *0.5
fwPlot, psdPlot, freqPlot = extractFreq(accPlot,fs,nperseg,noverlap)

# Plot
plt.rcParams['text.usetex'] = True
plt.rcParams['figure.dpi'] = 300
plt.rcParams['font.size'] = 16

# single figure with time series and spectrum
fig = plt.figure(figsize=(16, 2.5))
gs = mpl.gridspec.GridSpec(1, 2, width_ratios=[2, 1]) 
fig.suptitle('4 - Noise (N)', fontsize = 20, y = 1.05)

# Total Load Time Series
ax1 = fig.add_subplot(gs[0, 0])

# time history
ax1.plot(t, acc, color = 'grey', linewidth = 0.3, alpha = 0.5)
ax1.plot(t, accPlot, color = 'black', linewidth = 0.3)
ax1.set_xlabel('$t$ [$s$]', fontsize = 18)
ax1.set_ylabel('$a_{N}$ $[m/s^{2}]$')
ax1.set_xlim(t[0], t[-1])
ax1.tick_params(which ='major', direction = 'in', length = 10, labelsize = 14, pad = 5)  # Adjust major tick size
ax1.tick_params(which ='minor', direction = 'in', length = 5, pad = 5)  # Minor ticks are smaller
plt.grid(True)

# spectrum
ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(fwRef[:-1], psdRef[:-1], linewidth=1, color="grey")
ax2.plot(fwPlot[:-1], psdPlot[:-1], linewidth=1, color="black")
ax2.axvline(x=freqRef, color='grey', linestyle='--', linewidth=1.5)
ax2.axvline(x=freqPlot, color='black', linestyle='--', linewidth=1.5)
ax2.text(freqRef + 0.3, 0.35e-6, f'f = {freqRef:.2f} Hz', color='grey', fontsize=14)
ax2.text(freqPlot +0.3, 1.7e-6, f'f = {freqPlot:.2f} Hz', color='black', fontsize=14)
ax2.set_xlim(0, 60)
#ax2.set_ylim(0, 0.8e-6)
ax2.set_xlabel(r'$f$ [$Hz$]')
ax2.set_ylabel(r'$PSD$ [$m^2/s^3$]')
ax2.tick_params(which='major', direction='in', length=10, labelsize=14, pad=10)  # Adjust major tick size
ax2.tick_params(which='minor', direction='in', length=5, pad=10)  # Minor ticks are smaller
plt.grid(True)
plt.show()

In [ ]:
# PLOT EXAMPLE MISSING DATA
# example
keepPercMissing = np.random.randint(20, 80, size = 1)
keepPercMissing = keepPercMissing[0]/100
startMissing = int(keepPercMissing * len(acc))
aM = simulateMissing(acc, keepPercMissing)

accPlot = aM
def extractFreq(acc,fs,nperseg,noverlap):
	freqs, psd = welch(acc, fs, window = 'hann', nperseg = nperseg, noverlap = noverlap)
	peakInd = np.argmax(psd)
	extractedFreq = freqs[peakInd]
	return freqs, psd, extractedFreq
nperseg = 2**10
noverlap = nperseg *0.5
fwPlot, psdPlot, freqPlot = extractFreq(accPlot,fs,nperseg,noverlap)

# Plot
plt.rcParams['text.usetex'] = True
plt.rcParams['figure.dpi'] = 300
plt.rcParams['font.size'] = 16

# single figure with time series and spectrum
fig = plt.figure(figsize=(16, 2.5))
gs = mpl.gridspec.GridSpec(1, 2, width_ratios=[2, 1]) 
fig.suptitle('5 - Missing data (M)', fontsize = 20, y = 1.05)

# Total Load Time Series
ax1 = fig.add_subplot(gs[0, 0])

# time history
ax1.plot(t, acc, color = 'grey', linewidth = 0.3, alpha = 0.5)
ax1.plot(t, accPlot, color = 'black', linewidth = 0.3)
ax1.set_xlabel('$t$ [$s$]', fontsize = 18)
ax1.set_ylabel('$a_{M}$ $[m/s^{2}]$')
ax1.set_xlim(t[0], t[-1])
ax1.tick_params(which ='major', direction = 'in', length = 10, labelsize = 14, pad = 5)  # Adjust major tick size
ax1.tick_params(which ='minor', direction = 'in', length = 5, pad = 5)  # Minor ticks are smaller
plt.grid(True)

# spectrum
ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(fwRef[:-1], psdRef[:-1], linewidth=1, color="grey")
ax2.plot(fwPlot[:-1], psdPlot[:-1], linewidth=1, color="black")
ax2.axvline(x=freqRef, color='grey', linestyle='--', linewidth=1.5)
ax2.axvline(x=freqPlot, color='black', linestyle='--', linewidth=1.5)
ax2.text(freqRef + 0.3, 0.25e-6, f'f = {freqRef:.2f} Hz', color='grey', fontsize=14)
ax2.text(freqPlot +0.3, 0.35e-6, f'f = {freqPlot:.2f} Hz', color='black', fontsize=14)
ax2.set_xlim(0, 50)
#ax2.set_ylim(0, 0.8e-6)
ax2.set_xlabel(r'$f$ [$Hz$]')
ax2.set_ylabel(r'$PSD$ [$m^2/s^3$]')
ax2.tick_params(which='major', direction='in', length=10, labelsize=14, pad=10)  # Adjust major tick size
ax2.tick_params(which='minor', direction='in', length=5, pad=10)  # Minor ticks are smaller
plt.grid(True)
plt.show()

In [ ]:
# PLOT EXAMPLE CABLE DETACHMENT FULL
# example
startPoint = np.random.randint(20, acquisLength-20, size = 1)
startPoint = startPoint[0]
aCDF = simulateCableDetach(acc, startPoint, fs, duration = 900, frequency = 2, amplitude = 0.1, damping_factor = 0.05)

# PLOT EXAMPLE CABLE FULL + SPECTRUM
accPlot = aCDF
def extractFreq(acc,fs,nperseg,noverlap):
	freqs, psd = welch(acc, fs, window = 'hann', nperseg = nperseg, noverlap = noverlap)
	peakInd = np.argmax(psd)
	extractedFreq = freqs[peakInd]
	return freqs, psd, extractedFreq
nperseg = 2**10
noverlap = nperseg *0.5
fwPlot, psdPlot, freqPlot = extractFreq(accPlot,fs,nperseg,noverlap)

# Plot
plt.rcParams['text.usetex'] = True
plt.rcParams['figure.dpi'] = 300
plt.rcParams['font.size'] = 16

# single figure with time series and spectrum
fig = plt.figure(figsize=(16, 2.5))
gs = mpl.gridspec.GridSpec(1, 2, width_ratios=[2, 1]) 
fig.suptitle('6.1 - Cable detach (C)', fontsize = 20, y = 1.05)

# Total Load Time Series
ax1 = fig.add_subplot(gs[0, 0])

# time history
ax1.plot(t, acc, color = 'grey', linewidth = 0.3, alpha = 0.5)
ax1.plot(t, accPlot, color = 'black', linewidth = 0.3)
ax1.set_xlabel('$t$ [$s$]', fontsize = 18)
ax1.set_ylabel('$a_{C}$ $[m/s^{2}]$')
ax1.set_xlim(t[0], t[-1])
ax1.tick_params(which ='major', direction = 'in', length = 10, labelsize = 14, pad = 5)  # Adjust major tick size
ax1.tick_params(which ='minor', direction = 'in', length = 5, pad = 5)  # Minor ticks are smaller
plt.grid(True)

# spectrum
ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(fwRef[:-1], psdRef[:-1], linewidth=1, color="grey")
ax2.plot(fwPlot[:-1], psdPlot[:-1], linewidth=1, color="black")
ax2.axvline(x=freqRef, color='grey', linestyle='--', linewidth=1.5)
ax2.axvline(x=freqPlot, color='black', linestyle='--', linewidth=1.5)
ax2.text(freqRef + 0.5, 0.5e-5, f'f = {freqRef:.2f} Hz', color='grey', fontsize=14)
ax2.text(freqPlot +0.5, 2e-5, f'f = {freqPlot:.2f} Hz', color='black', fontsize=14)
ax2.set_xlim(0, 50)
#ax2.set_ylim(0, 0.8e-6)
ax2.set_xlabel(r'$f$ [$Hz$]')
ax2.set_ylabel(r'$PSD$ [$m^2/s^3$]')
ax2.yaxis.set_major_formatter(ticker.ScalarFormatter(useMathText=True))
ax2.ticklabel_format(style='sci', axis='y', scilimits=(0, 0))

ax2.tick_params(which='major', direction='in', length=10, labelsize=14, pad=10)  # Adjust major tick size
ax2.tick_params(which='minor', direction='in', length=5, pad=10)  # Minor ticks are smaller
plt.grid(True)
plt.show()

In [ ]:
# PLOT EXAMPLE CABLE DETACHMENT REATTACH
# example
startPoint = np.random.randint(20, acquisLength-20, size = 1)
startPoint = startPoint[0]
aCD = simulateCableDetachReattach(acc, startPoint, fs, duration = 2500, frequency = 2, amplitude = 0.2, damping_factor = 0.03)

# PLOT EXAMPLE CABLE + SPECTRUM
accPlot = aCD
def extractFreq(acc,fs,nperseg,noverlap):
	freqs, psd = welch(acc, fs, window = 'hann', nperseg = nperseg, noverlap = noverlap)
	peakInd = np.argmax(psd)
	extractedFreq = freqs[peakInd]
	return freqs, psd, extractedFreq
nperseg = 2**10
noverlap = nperseg *0.5
fwPlot, psdPlot, freqPlot = extractFreq(accPlot,fs,nperseg,noverlap)

# Plot
plt.rcParams['text.usetex'] = True
plt.rcParams['figure.dpi'] = 300
plt.rcParams['font.size'] = 16

# single figure with time series and spectrum
fig = plt.figure(figsize=(16, 2.5))
gs = mpl.gridspec.GridSpec(1, 2, width_ratios=[2, 1]) 
fig.suptitle('6.2 - Cable detach/reattach (C)', fontsize = 20, y = 1.05)

# Total Load Time Series
ax1 = fig.add_subplot(gs[0, 0])

# time history
ax1.plot(t, acc, color = 'grey', linewidth = 0.3, alpha = 0.5)
ax1.plot(t, accPlot, color = 'black', linewidth = 0.3)
ax1.set_xlabel('$t$ [$s$]', fontsize = 18)
ax1.set_ylabel('$a_{C}$ $[m/s^{2}]$')
ax1.set_xlim(t[0], t[-1])
ax1.tick_params(which ='major', direction = 'in', length = 10, labelsize = 14, pad = 5)  # Adjust major tick size
ax1.tick_params(which ='minor', direction = 'in', length = 5, pad = 5)  # Minor ticks are smaller
plt.grid(True)

# spectrum
ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(fwRef[:-1], psdRef[:-1], linewidth=1, color="grey")
ax2.plot(fwPlot[:-1], psdPlot[:-1], linewidth=1, color="black")
ax2.axvline(x=freqRef, color='grey', linestyle='--', linewidth=1.5)
ax2.axvline(x=freqPlot, color='black', linestyle='--', linewidth=1.5)
ax2.text(freqRef - 2.5, 0.25e-4, f'f = {freqRef:.2f} Hz', color='grey', fontsize=14)
ax2.text(freqPlot +0.5, 0.80e-4, f'f = {freqPlot:.2f} Hz', color='black', fontsize=14)
ax2.set_xlim(0, 50)
#ax2.set_ylim(0, 0.8e-6)
ax2.set_xlabel(r'$f$ [$Hz$]')
ax2.set_ylabel(r'$PSD$ [$m^2/s^3$]')
ax2.yaxis.set_major_formatter(ticker.ScalarFormatter(useMathText=True))
ax2.ticklabel_format(style='sci', axis='y', scilimits=(0, 0))

ax2.tick_params(which='major', direction='in', length=10, labelsize=14, pad=10)  # Adjust major tick size
ax2.tick_params(which='minor', direction='in', length=5, pad=10)  # Minor ticks are smaller
plt.grid(True)
plt.show()

In [ ]:
# PLOT EXAMPLE SPIKE
# example
aSp = acc
for i in range(10):
	startPoint = np.random.randint(20, acquisLength-20, size = 1)
	startPoint = startPoint[0]
	spikeMax = random.uniform(*spikeMaxRange)
	aSp = simulateSpike(aSp, startPoint, spikeMax)

accPlot = aSp
def extractFreq(acc,fs,nperseg,noverlap):
	freqs, psd = welch(acc, fs, window = 'hann', nperseg = nperseg, noverlap = noverlap)
	peakInd = np.argmax(psd)
	extractedFreq = freqs[peakInd]
	return freqs, psd, extractedFreq
nperseg = 2**10
noverlap = nperseg *0.5
fwPlot, psdPlot, freqPlot = extractFreq(accPlot,fs,nperseg,noverlap)

# Plot
plt.rcParams['text.usetex'] = True
plt.rcParams['figure.dpi'] = 300
plt.rcParams['font.size'] = 16

# single figure with time series and spectrum
fig = plt.figure(figsize=(16, 2.5))
gs = mpl.gridspec.GridSpec(1, 2, width_ratios=[2, 1]) 
fig.suptitle('7 - Spike (S)', fontsize = 20, y = 1.05)

# Total Load Time Series
ax1 = fig.add_subplot(gs[0, 0])

# time history
ax1.plot(t, acc, color = 'grey', linewidth = 0.3, alpha = 0.5)
ax1.plot(t, accPlot, color = 'black', linewidth = 0.3)
ax1.set_xlabel('$t$ [$s$]', fontsize = 18)
ax1.set_ylabel('$a_{S}$ $[m/s^{2}]$')
ax1.set_xlim(t[0], t[-1])
ax1.tick_params(which ='major', direction = 'in', length = 10, labelsize = 14, pad = 5)  # Adjust major tick size
ax1.tick_params(which ='minor', direction = 'in', length = 5, pad = 5)  # Minor ticks are smaller
plt.grid(True)

# spectrum
ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(fwRef[:-1], psdRef[:-1], linewidth=1, color="grey")
ax2.plot(fwPlot[:-1], psdPlot[:-1], linewidth=1, color="black")
ax2.axvline(x=freqRef, color='grey', linestyle='--', linewidth=1.5)
ax2.axvline(x=freqPlot, color='black', linestyle='--', linewidth=1.5)
ax2.text(freqRef + 0.3, 0.25e-6, f'f = {freqRef:.2f} Hz', color='grey', fontsize=14)
ax2.text(freqPlot + 0.3, 0.35e-6, f'f = {freqPlot:.2f} Hz', color='black', fontsize=14)
ax2.set_xlim(0, 50)
#ax2.set_ylim(0, 0.8e-6)
ax2.set_xlabel(r'$f$ [$Hz$]')
ax2.set_ylabel(r'$PSD$ [$m^2/s^3$]')
ax2.tick_params(which='major', direction='in', length=10, labelsize=14, pad=10)  # Adjust major tick size
ax2.tick_params(which='minor', direction='in', length=5, pad=10)  # Minor ticks are smaller
plt.grid(True)
plt.show()

#### Populate dataset with sensor faults

In [ ]:
# METHODS LIST
faultsMethods = {
	'drift': (simulateDrift, 0),
	'gain': (simulateGain, 1),
	'bias': (simulateBias, 2),
	'noise': (simulateNoise50, 3),
	'missing': (simulateMissing, 4),
	'cableDetach': (simulateCableDetach, 5),
	'cableDetachReattach': (simulateCableDetachReattach, 6),
	'spike': (simulateSpike, 7)
	}

fullFaultsMethods = {
	'drift': (simulateFullDrift, 0),
	'gain': (simulateFullGain, 1),
	'bias': (simulateFullBias, 2),
	'noise': (simulateFullNoise50, 3),
	'missing': (simulateFullMissing, 4),
	'cableDetach': (simulateFullMissing, 5),
}

In [ ]:
# SELECT PORTION OF SAMPLES TO CONTAMINATE
# timeseries
dateStartSFM = datetime(2021,7,1,0,0,0)
tHourly2 = pd.date_range(start = dateStartSFM, end = endDate, freq = ('{}{}'.format(intVal, offsetAlias)))
timeSimulation2 = tHourly2[~((tHourly2.month == 2) & (tHourly2.day == 29))]
indexStart = 0
accFilesSFM = accFilesD5[:] # D5
totalFiles = len(accFilesD5) # D5
nContaminated = int(len(accFilesSFM) * 0.3)
faultsCountEach = nContaminated // len(faultsMethods)

In [ ]:
# SELECT INDICES TO APPLY TO EACH FAULT
faultKeys = [
	'drift',
	'gain',
	'bias',
	'noise',
	'missing',
	'cableDetach',
	'cableDetachReattach',
	'spike'
]

usedIndices = set()  # tracks indices that are already contaminated
faultsLog = {}		# will map fileIndex -> (faultType, extraParamsIfNeeded)

def getRandomUnusedIndex(usedIndices, lowerLimit, upperLimit):
	"""
	Returns a random index that's not in usedIndices,
	or None if everything is used.
	"""
	candidates = [i for i in np.arange(lowerLimit, upperLimit, 1) if i not in usedIndices]
	if not candidates:
		return None
	return random.choice(candidates)

def getFaultLength():
	return random.randint(5, 25)

# classify which faults are single-instance
singleInstanceFaults = set(['spike', 'cableDetachReattach'])

for faultType in faultKeys:
	countApplied = 0
	while countApplied < faultsCountEach:
		# find a random unused index
		i = getRandomUnusedIndex(usedIndices, indexStart, (totalFiles+1))
		if i is None:
			print("We ran out of unused indices, just break")
			break
		
		# if it is single-instance, only contaminate that single file
		if faultType in singleInstanceFaults:
			# mark it used
			usedIndices.add(i)
			# store the fault type
			faultsLog[i] = (faultType, {})
			countApplied += 1
			faultsLog[i] = (faultType, {'partial': True})
		else:
			# multi-sample fault
			L = getFaultLength()
			endIndex = i + L
			# ensure not beyond last file and that none in that range are used
			if endIndex < totalFiles and not any(j in usedIndices for j in range(i, endIndex+1)):
				# mark them used
				for j in range(i, endIndex+1):
					usedIndices.add(j)
				faultsLog[i] = (faultType, {'partial': True})
				for j in range(i+1, endIndex+1):
					faultsLog[j] = (faultType, {'partial': False})
				
				countApplied += (L + 1)
			else:
				# if it doesn't fit, skip
				continue

In [ ]:
# LOOP FOR CONTAMINATION
for fileIndex, filePath in enumerate(accFilesD5):
	name = filePath.split('\\')[-1]
	numeric = name[3:8]
	newFilename = f"acc{numeric}-5.h5"
	newPath = outputFolder + "\\acceleration\\D5\\" + newFilename
	with h5py.File(filePath, 'r') as f:
		if 'acc' not in f:
			print(f"SKIPPED: 'acc' dataset not found in {filePath}") # check if h5 was created correctly
		acc = f['acc'][:]
		acc = acc[0]
		if fileIndex in faultsLog:
			faultType, faultParams = faultsLog[fileIndex]
			# apply singleIndexFaults
			if faultType in singleInstanceFaults:
				simulateFunc, _ = faultsMethods[faultType]
				if faultType == 'spike':
					faultedAcc = acc
					nSp = np.random.randint(0, 25, size = 1)
					for i in range(nSp[0]):
						startPoint = np.random.randint(begLimit, acquisLength-endLimit, size = 1)
						startPoint = startPoint[0]
						spikeMax = random.uniform(*spikeMaxRange)
						faultedAcc = simulateFunc(faultedAcc, startPoint, spikeMax)
				if faultType == 'cableDetachReattach':
					startPoint = np.random.randint(begLimit, acquisLength-endLimit-(duration/fs), size = 1)
					startPoint = startPoint[0]
					faultedAcc = simulateFunc(	acc, startPoint, fs, 
								  				duration = 2500, frequency = random.uniform(*frequencyRange), 
												amplitude = random.uniform(*amplitudeRange), damping_factor = damping_factor)
			# apply multiIndexFaults
			else:
				startPoint = np.random.randint(begLimit, acquisLength-endLimit, size = 1)
				startPoint = startPoint[0]
				if faultType == 'drift':
					if faultParams.get('partial', True):
						simulateFunc, _ = faultsMethods[faultType]
						# simulate drift
						faultedAcc, driftShift = simulateFunc(acc, startPoint, fs = fs, scale = random.uniform(*scaleDriftRange)) # start Point is provided in seconds
					elif faultParams.get('partial', False):
						simulateFullFunc, _ = fullFaultsMethods[faultType]
						# simulate shift after drift
						faultedAcc = simulateFunc(acc, driftShift)
				elif faultType == 'gain':
					if faultParams.get('partial', True):
						simulateFunc, _ = faultsMethods[faultType]
						# simulate gain
						faultedAcc, factorGain = simulateFunc(acc, startPoint, fs, factorGain = random.uniform(*factorGainRange))
					elif faultParams.get('partial', False):
						simulateFullFunc, _ = fullFaultsMethods[faultType]
						# simulate full gain 
						faultedAcc = simulateFunc(acc, factorGain)
				elif faultType == 'bias':
					if faultParams.get('partial', True):
						simulateFunc, _ = faultsMethods[faultType]
						# simulate bias
						faultedAcc, biasMag = simulateFunc(acc, startPoint, fs, biasMag = random.uniform(*biasMagRange))
					elif faultParams.get('partial', False):
						simulateFullFunc, _ = fullFaultsMethods[faultType]
						# simulate full bias 
						faultedAcc = simulateFunc(acc, biasMag)
				elif faultType == 'noise':
					if faultParams.get('partial', True):
						simulateFunc, _ = faultsMethods[faultType]
						# simulate noise
						faultedAcc, noise = simulateFunc(acc, startPoint, fs, noiseStd = random.uniform(*noiseStdRange), noiseFreq = 50)
					elif faultParams.get('partial', False):
						simulateFullFunc, _ = fullFaultsMethods[faultType]
						# simulate full noise 
						faultedAcc = simulateFunc(acc, noise)
				elif faultType == 'missing':
					if faultParams.get('partial', True):
						simulateFunc, _ = faultsMethods[faultType]
						# simulate missign
						faultedAcc = simulateFunc(acc, keepPercMissing = random.uniform(*keepPercMissingRange))
					elif faultParams.get('partial', False):
						simulateFullFunc, _ = fullFaultsMethods[faultType]
						# simulate full missing 
						faultedAcc = simulateFunc(acc)
				elif faultType == 'cableDetach':
					if faultParams.get('partial', True):
						simulateFunc, _ = faultsMethods[faultType]
						# simulate cable detach
						startPoint = np.random.randint(begLimit, acquisLength-endLimit-(duration/fs), size = 1)
						startPoint = startPoint[0]
						faultedAcc = simulateFunc(	acc, startPoint, fs, duration = 2500, 
													frequency = random.uniform(*frequencyRange), 
													amplitude = random.uniform(*amplitudeRange), 
													damping_factor = 0.05)
					elif faultParams.get('partial', False):
						simulateFullFunc, _ = fullFaultsMethods[faultType]
						# simulate full deteach 
						faultedAcc = simulateFunc(acc)
			# Save
			with h5py.File(newPath, 'w') as f_new:
				f_new.create_dataset('acc', data = faultedAcc)
		else:
			pass